# Data integrity, step by step

Runs the clean pipeline on the collected files and checks every hop from A to B:
did a step lose or add rows, duplicate a key, change a value it should carry over,
or turn a value into null? The checks are `compare`, `changed_rows` and `disagree` in
`money_printer_research.clean_data.integrity`.

Nothing is written to disk. Run `uv run collect_data` first; the last section also
checks the CSVs that `uv run clean_data` wrote.

In [1]:
import polars as pl

from money_printer_research.clean_data import clean, extract_features
from money_printer_research.clean_data.integrity import (
    changed_rows,
    compare,
    disagree,
    log_report,
    trace,
)
from money_printer_research.clean_data.match_stat import MATCH_STAT_STEPS, clean_match_stat
from money_printer_research.clean_data.player_stat import PLAYER_STAT_STEPS
from money_printer_research.clean_data.read import (
    read_match_info,
    read_match_stat,
    read_player_stat,
)
from money_printer_research.config import settings
from money_printer_research.cool_stuff import told
from money_printer_research.features import reason_counts

pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(30)

polars.config.Config

## Run the pipeline

The raw files, the cleaned Understat schedule and player stats on their own, then the
whole of `clean` and `extract_features`.

In [2]:
raw_schedule = read_match_stat(settings).expect("read_match_stat")
raw_players = read_player_stat(settings).expect("read_player_stat")
raw_games = read_match_info(settings).expect("read_match_info")

schedule = clean_match_stat(raw_schedule).expect("clean_match_stat")
cleaned = clean(settings).expect("clean")
features = extract_features(cleaned).expect("extract_features")

## Each morphism, logged

`trace(*steps, key=...)` turns every step into a Kleisli arrow that also writes, at that
morphism, what happened to the rows, the keys and the nulls (`at_morphism`, a Writer
over `Result`). The arrows compose with `>>` like any other `K`, so this is the same
pipeline as `clean_match_stat`, with a log. One row per entry, the input first:
`more_nulls` names the columns that gained nulls there.

In [3]:
logged_schedule = trace(*MATCH_STAT_STEPS, key="game_id")(told(raw_schedule)).expect("schedule")
assert logged_schedule.value.equals(schedule)
log_report(logged_schedule)

morphism,shape,nulls,more_nulls,lost_keys,new_keys,dup_keys,changed,new_nulls
str,list[i64],i64,list[null],i64,i64,i64,i64,i64
null,"[21690, 20]",404,null,null,null,null,null,null
"""_keep_played_matches""","[21589, 20]",0,[],101,0,0,0,0
"""check(UnderstatScheduleRaw)""","[21589, 20]",0,[],0,0,0,0,0
"""_add_season_start""","[21589, 21]",0,[],0,0,0,0,0
"""check_aliases""","[21589, 21]",0,[],0,0,0,0,0
"""norm_team_names""","[21589, 21]",0,[],0,0,0,43178,0
"""check(ScheduleCols)""","[21589, 21]",0,[],0,0,0,0,0


In [4]:
# Each entry keeps the frame it logged: here, what the first morphism returned.
for entry in logged_schedule.log:
    print(entry.morphism, entry.shape)
logged_schedule.log[1].a.head()

None (21690, 20)
_keep_played_matches (21589, 20)
check(UnderstatScheduleRaw) (21589, 20)
_add_season_start (21589, 21)
check_aliases (21589, 21)
norm_team_names (21589, 21)
check(ScheduleCols) (21589, 21)


league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
str,i64,str,i64,i64,i64,datetime[μs],i64,i64,str,str,str,str,i64,i64,f64,f64,bool,bool,str
"""ENG-Premier League""",1415,"""2014-08-16 Arsenal-Crystal Pal…",1,2014,4755,2014-08-16 17:30:00,83,78,"""Arsenal""","""Crystal Palace""","""CRY""","""ARS""",2,1,1.55411,0.158151,true,true,"""https://understat.com/match/47…"
"""ENG-Premier League""",1415,"""2014-08-16 Leicester-Everton""",1,2014,4750,2014-08-16 15:00:00,75,72,"""Leicester""","""Everton""","""EVE""","""LEI""",2,2,1.2783,0.613273,true,true,"""https://understat.com/match/47…"
"""ENG-Premier League""",1415,"""2014-08-16 Manchester United-S…",1,2014,4749,2014-08-16 12:45:00,89,84,"""Manchester United""","""Swansea""","""SWA""","""MUN""",1,2,1.16635,0.278076,true,true,"""https://understat.com/match/47…"
"""ENG-Premier League""",1415,"""2014-08-16 Queens Park Rangers…",1,2014,4751,2014-08-16 15:00:00,202,91,"""Queens Park Rangers""","""Hull""","""HUL""","""QPR""",0,1,1.90067,1.11757,true,true,"""https://understat.com/match/47…"
"""ENG-Premier League""",1415,"""2014-08-16 Stoke-Aston Villa""",1,2014,4752,2014-08-16 15:00:00,85,71,"""Stoke""","""Aston Villa""","""AVL""","""STO""",0,1,0.423368,0.909774,true,true,"""https://understat.com/match/47…"


In [5]:
logged_players = trace(*PLAYER_STAT_STEPS, key=["game_id", "player_id"])(told(raw_players))
log_report(logged_players.expect("player_stat"))

morphism,shape,nulls,more_nulls,lost_keys,new_keys,dup_keys,changed,new_nulls
str,list[i64],i64,list[null],i64,i64,i64,i64,i64
null,"[630291, 24]",0,null,null,null,null,null,null
"""_add_season_start""","[630291, 25]",0,[],0,0,62,146,0
"""check_aliases""","[630291, 25]",0,[],0,0,62,146,0
"""norm_team_names""","[630291, 25]",0,[],0,0,62,630561,0
"""check(PlayerStatCols)""","[630291, 25]",0,[],0,0,62,146,0


## Every hop

One row per step. `lost_keys` and `new_keys` are keys on only one side, `dup_keys` keys
repeated in the output, `changed` values that differ, `new_nulls` values that became
null. All of them should be 0, with these exceptions:

- **Transfermarkt games → match_leagues** loses the games without a date, on purpose.
- **schedule → cleaned_matches** and **cleaned_matches → match_features** only add
  columns, so anything other than 0 there is damage.

Team names are left out of the first two hops because `norm_team_names` rewrites them
on purpose, and Transfermarkt columns are renamed to their clean names first.

In [6]:
TM_RENAMES = {
    "game_id": "tm_game_id",
    "home_club_goals": "home_goals",
    "away_club_goals": "away_goals",
    "home_club_formation": "home_formation",
    "away_club_formation": "away_formation",
}
TEAM_NAMES = {"home_team", "away_team", "team"}


def without_teams(before: pl.DataFrame, after: pl.DataFrame) -> list[str]:
    return [c for c in before.columns if c in after.columns and c not in TEAM_NAMES]


played = raw_schedule.filter(pl.col("is_result"))
games = raw_games.rename(TM_RENAMES)
key_players = ["game_id", "player_id"]

HOPS = {
    "raw schedule (played) -> schedule": (
        played,
        schedule,
        "game_id",
        without_teams(played, schedule),
    ),
    "raw players -> player_stat": (
        raw_players,
        cleaned.player_stat,
        key_players,
        without_teams(raw_players, cleaned.player_stat),
    ),
    "Transfermarkt games -> match_leagues": (games, cleaned.match_leagues, "tm_game_id", None),
    "schedule -> cleaned_matches": (schedule, cleaned.cleaned_matches, "game_id", None),
    "cleaned_matches -> match_features": (
        cleaned.cleaned_matches,
        features.match_features,
        "game_id",
        None,
    ),
}
report = pl.concat(
    [
        compare(before, after, key, cols, step=step)
        for step, (before, after, key, cols) in HOPS.items()
    ]
)
report

step,rows_before,rows_after,lost_keys,new_keys,dup_keys,changed,new_nulls
str,i64,i64,i64,i64,i64,i64,i64
"""raw schedule (played) -> sched…",21589,21589,0,0,0,0,0
"""raw players -> player_stat""",630291,630291,0,0,62,146,0
"""Transfermarkt games -> match_l…",28140,28140,0,0,0,0,0
"""schedule -> cleaned_matches""",21589,21589,0,0,0,0,0
"""cleaned_matches -> match_featu…",21589,21589,0,0,0,0,0


## Look at one hop

Pick a step from the table above to see the cells behind its `changed` and
`new_nulls` counts.

In [7]:
before, after, key, cols = HOPS["schedule -> cleaned_matches"]
changed_rows(before, after, key, cols).head(30)

game_id,column,before,after
i64,str,str,str


## Do the two sources agree?

Once an Understat match is joined to its Transfermarkt game, both should have the same
score. A few rows here are a data error in one source; many mean the join paired the
wrong games.

In [8]:
GOALS = {"home_goals": "tm_home_goals", "away_goals": "tm_away_goals"}
mismatch = disagree(cleaned.cleaned_matches, GOALS)
matched = cleaned.cleaned_matches["tm_game_id"].is_not_null().sum()
print(f"{mismatch.height} of {matched} matched games disagree on the score")
mismatch.select("game_id", "date", "home_team", "away_team", *GOALS, *GOALS.values())

4 of 21589 matched games disagree on the score


game_id,date,home_team,away_team,home_goals,away_goals,tm_home_goals,tm_away_goals
i64,datetime[μs],str,str,i64,i64,i64,i64
4238,2017-04-16 00:00:00,"""sc bastia""","""lyon""",0,3,0,0
27866,2024-12-14 14:30:00,"""union berlin""","""bochum""",1,1,0,2
945,2016-08-28 22:45:00,"""sassuolo""","""pescara""",2,1,0,3
14117,2020-09-19 18:45:00,"""verona""","""roma""",0,0,3,0


## Missing table

`features.missing` (the `missing` frame, `missing.csv` on disk): one row per null cell
of match_features, with its side and reason. First the count per reason, then the
cells behind one reason.

In [9]:
assert features.missing.height == features.match_features.null_count().sum_horizontal().item()
reason_counts(features.missing)

reason,cells,matches
str,u32,u32
"""too_few_minutes""",87462,12755
"""plain_shape""",27220,18313
"""no_formation""",777,56
"""no_lineup""",230,2
"""no_next_game""",112,64
"""short_lineup""",60,11


In [10]:
features.missing.filter(pl.col("reason") == "other").head(30)

game_id,column,side,reason
i64,str,str,str


## In memory → CSV on disk

Reads back each CSV that `uv run clean_data` wrote and compares it with the frame just
built. A value that does not survive the round trip, such as a date written in a form
`read_csv` cannot parse back, shows as a new null. Skips files that are not there yet.

In [11]:
KEYS = {
    "match_leagues": "tm_game_id",
    "clubs": "club_id",
    "cleaned_matches": "game_id",
    "player_stat": ["game_id", "player_id"],
    "elo": ["match_date", "home_team", "away_team"],
    "fatigue": "game_id",
    "schedule": "game_id",
    "players": "game_id",
    "match_features": "game_id",
    "missing": ["game_id", "column"],
}
frames = {**vars(cleaned), **vars(features)}
out_dir = settings.clean.out_dir
on_disk = []
for name, key in KEYS.items():
    path = out_dir / f"{name}.csv"
    if not path.exists():
        print("missing", path)
        continue
    saved = pl.read_csv(path, infer_schema_length=None, try_parse_dates=True)
    on_disk.append(compare(frames[name], saved, key, step=f"{name} -> {path.name}"))
pl.concat(on_disk) if on_disk else None

step,rows_before,rows_after,lost_keys,new_keys,dup_keys,changed,new_nulls
str,i64,i64,i64,i64,i64,i64,i64
"""match_leagues -> match_leagues…",28140,28140,0,0,0,0,0
"""clubs -> clubs.csv""",167,167,0,0,0,0,0
"""cleaned_matches -> cleaned_mat…",21589,21589,0,0,0,0,0
"""player_stat -> player_stat.csv""",630291,630291,0,0,62,146,0
"""elo -> elo.csv""",21589,21589,0,0,0,17134,0
"""fatigue -> fatigue.csv""",21587,21587,0,0,0,0,0
"""schedule -> schedule.csv""",21589,21589,0,0,0,0,0
"""players -> players.csv""",21587,21587,0,0,0,0,0
"""match_features -> match_featur…",21589,21589,0,0,0,17134,0
